# Infaction — LOCKED Political + Geographic Placement
Runs the **unchanged** Faction, Naming, Zone and Location generator (v0.4.3 city edition), then adds approved political 50:50 and geographic 70:30 placement. This notebook does not modify locked input selection or source Book1. No physical cutoff, forbidden pairs, quota, or fallback.
Source: [master specification](https://docs.google.com/document/d/15AVs2_mh66o-M8YnetGeLZqB2HuXRm04ZqIhNWWHT6s/edit) and [placement engine](https://github.com/playrenathea/rantau/blob/main/infaction_placement_locked.py).


In [ ]:
import urllib.request, json, contextlib, io, types
ROOT='https://raw.githubusercontent.com/playrenathea/rantau/main/'
def get_text(path): return urllib.request.urlopen(ROOT+path,timeout=30).read().decode('utf-8')
source=json.loads(get_text('notebooks/Infaction_Faction_Naming_Zone_Location_Generator_v0.4.3_City_Experiment_Revised.ipynb'))
ns={'__name__':'__main__'}
for i in (1,2,3,5,8,11):
    with contextlib.redirect_stdout(io.StringIO()):
        exec(compile(''.join(source['cells'][i]['source']),str(i),'exec'),ns)
placement=types.ModuleType('infaction_placement_locked')
exec(compile(get_text('infaction_placement_locked.py'),'infaction_placement_locked.py','exec'),placement.__dict__)
neutral_data=json.loads(get_text('data/infaction_neutral_static_v2_locked.json'))
print('Neutral keys:',list(neutral_data.keys()))
records=neutral_data['records']
neutral={r['place']:r['neutral_affinity'] for r in records}
assert len(neutral)==110
print('Ready:',placement.VERSION)


In [ ]:
SEED=8100000
world=ns['generate_world_with_city_zones'](SEED)
result=placement.place_world(world,ns['ZONE_DATA'],ns['LOCATION_DATA'],neutral,seed=SEED)
print('Factions:',[(f['id'],f['types'],f['base']) for f in result['factions']])
for zone,locs in result['by_zone'].items():
    print(f'{zone:24s}',result['political'][zone]['label'], '=>', ', '.join(locs) or '(empty)')
print('Outside:',result['outside'])
print('Stage1:',sum(v['stage']==1 for v in result['placement'].values()),'Stage2 Inside:',sum(v['stage']==2 and v['zone'] is not None for v in result['placement'].values()),'Outside:',len(result['outside']))


In [ ]:
# Small regression batch; avoid slow full-source QA cells.
from collections import Counter
N=100
counts=Counter()
for seed in range(8100000,8100000+N):
    w=ns['generate_world_with_city_zones'](seed)
    out=placement.place_world(w,ns['ZONE_DATA'],ns['LOCATION_DATA'],neutral,seed)
    assert len(out['placement'])==len(out['locations'])
    assert all(v['zone'] is None or v['zone'] in out['zones'] for v in out['placement'].values())
    assert all(v['stage'] in (1,2) for v in out['placement'].values())
    counts.update({'worlds':1,'stage1':sum(v['stage']==1 for v in out['placement'].values()),'inside2':sum(v['stage']==2 and v['zone'] is not None for v in out['placement'].values()),'outside':len(out['outside'])})
print('Regression averages:',{k:round(v/N,2) for k,v in counts.items() if k!='worlds'})


In [ ]:
# Export complete seed result for downstream MVP mapping (opt-in; no UI/runtime replacement).
from pathlib import Path
Path(f'infaction_world_placement_{SEED}.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
print('Export complete')
